In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/holdings_2024_Q1.csv", low_memory=False)

# What percentage of unique CUSIPs are ETFs?
etf_keywords = ["ETF", "TRUST", "FUND", "INDEX", "ISHARES", 
                 "VANGUARD", "SPDR", "INVESCO", "BLACKROCK"]

mask = df["name"].str.upper().str.contains("|".join(etf_keywords), na=False)
print("ETF/Fund rows:", mask.sum())
print("Stock rows:", (~mask).sum())
print("ETF percentage:", f"{mask.sum()/len(df)*100:.1f}%")

# Show unique ETF-like names
etf_names = df[mask]["name"].unique()
print("\nSample ETF names:")
print(etf_names[:20])

ETF/Fund rows: 83453
Stock rows: 238205
ETF percentage: 25.9%

Sample ETF names:
<StringArray>
[         'SPDR S&P 500 ETF TR',   'SPDR SER TR SPDR BLOOMBERG',
      'MEDICAL PPTYS TRUST INC',                   'ISHARES TR',
                  'SPDR SER TR', 'VANGUARD INTL EQUITY INDEX F',
  'ABRDN JAPAN EQUITY FUND INC', 'BLACKROCK ENHANCED GLOBAL DI',
 'BLACKROCK CR ALLOCATION INCO', 'BLACKROCK HEALTH SCIENCES TR',
  'BLACKROCK ENHANCED INTL DIV',    'BLACKROCK MUNIVEST FD INC',
 'BLACKROCK ENHANCED GOVT FD I', 'BLACKROCK MUN TARGET TERM TR',
 'BLACKROCK HEALTH SCIENCES TE', 'BLACKROCK SCIENCE & TECHNOLO',
 'BLACKROCK INNOVATION AND GRW', 'BLACKROCK CAP ALLOCATION TER',
    'SRH TOTAL RETURN FUND INC',                'INVESCO BD FD']
Length: 20, dtype: str


In [2]:
import requests

HEADERS = {"User-Agent": "gvip-predictor bencheng18@gmail.com"}
response = requests.get(
    "https://www.sec.gov/Archives/edgar/data/1067983/000119312526226661/53405.xml",
    headers=HEADERS
)

from lxml import etree
root = etree.fromstring(response.content)
namespace = {"ns": "http://www.sec.gov/edgar/document/thirteenf/informationtable"}

for info in root.findall("ns:infoTable", namespace)[:5]:
    print({
        "name": info.findtext("ns:nameOfIssuer", namespaces=namespace),
        "titleOfClass": info.findtext("ns:titleOfClass", namespaces=namespace),
    })

{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}


In [3]:
# Use a wealth manager that likely holds ETFs
response = requests.get(
    "https://data.sec.gov/submissions/CIK0001602119.json",
    headers=HEADERS
)
data = response.json()

import pandas as pd
filings = pd.DataFrame(data["filings"]["recent"])
df_13f = filings[filings["form"] == "13F-HR"].iloc[0]

acc = df_13f["accessionNumber"].replace("-", "")
cik = 1602119
xml_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/"

# Get the holdings XML
import requests, re
index_response = requests.get(
    f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{df_13f['accessionNumber']}-index.htm",
    headers=HEADERS
)
xml_files = re.findall(r'href="([^"]+\.xml)"', index_response.text)
holdings_xml = [f for f in xml_files if "primary_doc" not in f]

if holdings_xml:
    filename = holdings_xml[0].split("/")[-1]
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{filename}"
    response = requests.get(url, headers=HEADERS)
    root = etree.fromstring(response.content)
    namespace = {"ns": "http://www.sec.gov/edgar/document/thirteenf/informationtable"}
    
    seen = set()
    for info in root.findall("ns:infoTable", namespace):
        title = info.findtext("ns:titleOfClass", namespaces=namespace)
        name = info.findtext("ns:nameOfIssuer", namespaces=namespace)
        if title not in seen:
            print(f"{title}: {name}")
            seen.add(title)

COM CL A1: ACCEL ENTERTAINMENT INC
COM: CECO ENVIRONMENTAL CORP
COM SHS: CENTURI HOLDINGS INC
CL A COM: KURA SUSHI USA INC


In [1]:
import pandas as pd
import glob

files = sorted(glob.glob("../data/raw/holdings_*.csv"))

all_cusips = set()
for f in files:
    df = pd.read_csv(f, low_memory=False)
    all_cusips.update(df["cusip"].dropna().unique())

cusip_list = list(all_cusips)
print("Total CUSIPs:", len(cusip_list))

# Check for malformed CUSIPs
cusip_series = pd.Series(cusip_list)
print("\nCUSIP length distribution:")
print(cusip_series.str.len().value_counts().sort_index())

# Show examples of non-standard length CUSIPs
weird = cusip_series[cusip_series.str.len() != 9]
print("\nNon-standard CUSIPs:")
print(weird.values[:20])

Total CUSIPs: 27360

CUSIP length distribution:
9    27360
Name: count, dtype: int64

Non-standard CUSIPs:
<StringArray>
[]
Length: 0, dtype: str
